# 实验三：说话人验证与说话人辨认（50%）

在实验二中，我们观察了 speaker embeddings 在低维空间中的分布，并初步分析了不同说话人、不同扰动条件下 embedding 的变化。实验三将进一步使用这些 embeddings 完成两个典型的说话人识别任务：

- 说话人验证（Speaker Verification）：给定一个注册说话人和一段测试语音，系统需要判断二者是否属于同一个人。
- 说话人辨认（Speaker Identification）：给定一段测试语音，系统需要从一组已注册说话人中判断该语音属于谁。

本实验将引入三个核心概念：

- Enrollment（注册语音）：用于构建某个说话人的模板（speaker template），多个 enrollment utterance 对应一个 enrollment speaker。
- Trial（测试对）：由 enrollment speaker、test utterance 和 same/different label 组成。
- Threshold：决策阈值，用于将连续的 cosine similarity score 转换为 same/different 判断。


> Speaker Verification 关注的问题是：“这是不是某个人”。
>
> Speaker Identification 关注的问题是：“这是谁”。
>
> 二者都可以基于 speaker embedding 实现，但任务定义、决策方式和评测指标并不相同。

## Enrollment 与 Trial

在说话人识别任务中，系统通常不能直接“凭空”判断一段语音属于谁，而是需要先知道候选说话人的声音特征。因此，我们需要先进行 **enrollment**。

### Enrollment

**Enrollment** 指的是注册阶段。对于每个已知说话人，我们取若干条该说话人的语音，提取 speaker embeddings，并将这些 embeddings 聚合成一个 speaker template。这个 template 可以理解为该说话人在 embedding 空间中的“声音指纹”。

例如，对说话人 `S0251`，我们可能有 3 条 enrollment utterances：

| utt_id | spk_id | path |
|---|---|---|
| BAC009S0251W0458 | S0251 | wav/S0251/BAC009S0251W0458.wav |
| BAC009S0251W0349 | S0251 | wav/S0251/BAC009S0251W0349.wav |
| BAC009S0251W0248 | S0251 | wav/S0251/BAC009S0251W0248.wav |

我们分别提取这 3 条语音的 embedding，然后取平均，得到 `S0251` 的 enrollment template。

#### Speaker Template

对于说话人 $s$，假设其共有 $K$ 条 enrollment utterances。第 $i$ 条 enrollment utterance 提取到的 L2-normalized speaker embedding 记为 $\hat{z}_{s,i}$。

该说话人的 speaker template 定义为所有 enrollment embeddings 的平均向量，并再次进行 L2 normalization：

$$
e_s =
\frac{
\frac{1}{K}\sum_{i=1}^{K}\hat{z}_{s,i}
}{
\left\|
\frac{1}{K}\sum_{i=1}^{K}\hat{z}_{s,i}
\right\|_2
}
$$

### Trial

**Trial** 指的是一次测试样本或一次判断请求。不同任务中，trial 的形式不同。

### Speaker Verification 中的 Trial

Speaker Verification 关注的问题是：

> 这段语音是不是某个人说的？

在 verification 任务中，enrollment 同样用于构建每个 known speaker 的 template。但测试时，不再是在所有说话人中选一个，而是给定一个具体的 enrollment speaker 和一条 test utterance，判断二者是否属于同一个人。

一个 verification trial 通常长这样：

| enroll_spk_id | test_utt_id | label |
|---|---|---|
| S0251 | BAC009S0251W0480 | 1 |
| S0251 | BAC009S0160W0331 | 0 |

其中：

- `enroll_spk_id` 表示要验证的目标说话人；
- `test_utt_id` 表示待测试语音；
- `label=1` 表示 same speaker，即测试语音确实来自 `enroll_spk_id`；
- `label=0` 表示 different speaker，即测试语音来自其他人。


系统会计算：

```text
score = cosine(template[enroll_spk_id], embedding[test_utt_id])
```

然后根据 threshold 做判断：

```text
score >= threshold  -> same speaker
score < threshold   -> different speaker
```

#### 相似度分数


给定说话人 $s$ 的 template $e_s$，以及测试语音 $t$ 的 speaker embedding $\hat{z}_t$，二者的相似度分数定义为：

$$
\operatorname{score}(s,t)
=
\cos(e_s,\hat{z}_t)
=
\frac{e_s^\top \hat{z}_t}
{\|e_s\|_2 \|\hat{z}_t\|_2}
$$

如果 $e_s$ 和 $\hat{z}_t$ 都已经做过 L2 normalization，则有：

$$
\operatorname{score}(s,t)
=
e_s^\top \hat{z}_t
$$

#### Verification 决策

在 speaker verification 中，系统需要判断测试语音 $t$ 是否属于 enrollment speaker $s$。

给定阈值 $\tau$，决策规则为：

$$
\hat{y}
=
\begin{cases}
1, & \operatorname{score}(s,t) \ge \tau \\
0, & \operatorname{score}(s,t) < \tau
\end{cases}
$$

其中，$1$ 表示 same speaker，$0$ 表示 different speaker。

---

### Speaker Identification 中的 Trial

Speaker Identification 关注的问题是：

> 这段语音是谁说的？

在 identification 任务中，enrollment 阶段会为每个 known speaker 构建一个 template。测试时，给定一条 test utterance，系统将它的 embedding 与所有 enrolled speakers 的 templates 计算相似度，并选择相似度最高的 speaker 作为预测结果。

一个 identification trial 可以理解为：

| test_utt_id | true_spk_id | path | is_known |
|---|---|---|---|
| BAC009S0251W0448 | S0251 | wav/S0251/BAC009S0251W0448.wav | 1 |

系统需要判断这条语音最像哪个 enrollment speaker。

如果是 closed-set identification，系统假设测试语音一定来自已注册说话人之一，因此直接选择相似度最高的 speaker。

如果是 open-set identification，测试语音可能来自 unknown speaker。此时系统除了选择最相似的 enrolled speaker，还需要判断最高相似度是否足够高。如果最高相似度低于 threshold，则可以拒识为 unknown。

#### Identification 决策

在 speaker identification 中，系统需要从已注册说话人集合中判断测试语音 $t$ 属于谁。

##### 闭集 Identification

闭集 identification 假设测试语音一定来自某个已注册说话人。因此，系统直接选择得分最高的说话人：

$$
\hat{s}
=
\arg\max_s \operatorname{score}(s,t)
$$

##### 开放集 Identification

开放集 identification 允许测试语音来自未注册说话人。此时，系统不仅需要找到最高分的说话人，还需要判断最高分是否超过 identification threshold $\tau_{id}$：

$$
\hat{s}
=
\begin{cases}
\arg\max_s \operatorname{score}(s,t), & \max_s \operatorname{score}(s,t) \ge \tau_{id} \\
\mathrm{unknown}, & \max_s \operatorname{score}(s,t) < \tau_{id}
\end{cases}
$$

其中，$\mathrm{unknown}$ 表示系统拒识，认为该测试语音不属于任何已注册说话人。

---

## 二者的差异

| 任务 | 问题 | Enrollment 的作用 | Trial 的形式 | 输出 |
|---|---|---|---|---|
| Speaker Verification | 这是不是某个人？ | 为目标说话人构建 template | enroll speaker + test utterance | same / different |
| Speaker Identification | 这是谁？ | 为每个候选说话人构建 template | 一条 test utterance | speaker id 或 unknown |



## 使用到的 metadata

- `protocols/enroll.csv`：注册语音列表。用于为每个已知说话人构建 enrollment template。
- `protocols/identification_test.csv`：说话人辨认测试集。包含 closed-set 测试语音和 open-set unknown 测试语音。
- `protocols/verification_trials_dev.csv`：说话人验证开发集 trials。用于扫描 threshold，并分析不同阈值下的 FAR / FRR。
- `protocols/verification_trials_test.csv`：说话人验证测试集 trials。用于在选定 threshold 后评估最终 verification 性能。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

PROJECT_ROOT = Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "aishell_mini"
OUT_DIR = PROJECT_ROOT / "outputs"

enroll_df = pd.read_csv(DATA_DIR / "protocols" / "enroll.csv")
id_test_df = pd.read_csv(DATA_DIR / "protocols" / "identification_test.csv")
utterances_df = pd.read_csv(DATA_DIR / "metadata" / "utterances.csv")
dev_trials = pd.read_csv(DATA_DIR / "protocols" / "verification_trials_dev.csv")
test_trials = pd.read_csv(DATA_DIR / "protocols" / "verification_trials_test.csv")

display(enroll_df.head())
display(id_test_df.head())
display(dev_trials.head())

print("num enrollment speakers:", enroll_df["spk_id"].nunique())
print("num identification test utterances:", len(id_test_df))
print("dev trial labels:")
print(dev_trials["label"].value_counts())


In [ ]:
# 加载 CN-Celeb ECAPA-TDNN 预训练模型
import torch
import torchaudio
import torch.nn.functional as F
from speechbrain.inference.speaker import EncoderClassifier

device = "cuda" if torch.cuda.is_available() else "cpu"

classifier = EncoderClassifier.from_hparams(
    source="LanceaKing/spkrec-ecapa-cnceleb", 
    savedir="ckpt/spkrec-ecapa-cnceleb", 
    run_opts={"device": device})

@torch.no_grad()
def extract_embedding_from_path(path):
    wav, sr = torchaudio.load(path)
    wavs = wav.squeeze(0).unsqueeze(0).to(device)
    emb = classifier.encode_batch(wavs).squeeze().detach().cpu()
    emb = F.normalize(emb, dim=0)
    return emb

# 构建 embedding cache
embedding_cache = {}

def get_embedding_by_path(path):
    path = str(path)
    if path not in embedding_cache:
        embedding_cache[path] = extract_embedding_from_path(DATA_DIR / path)
    return embedding_cache[path]

## Enrollment Templates

In [ ]:
# 构建 enrollment templates
speaker_templates = {}

for spk_id, group in enroll_df.groupby("spk_id"):
    embs = []

    for _, row in group.iterrows():
        emb = get_embedding_by_path(row["path"])
    # YOUR CODE HERE

    speaker_templates[spk_id] = template

print("num templates:", len(speaker_templates))
print("template dim:", next(iter(speaker_templates.values())).shape)


## 回答问题：
Q1. Speaker Templates
- 如果每个 speaker 只有一条 enrollment，template 是否仍然成立？
- 多条 enrollment 平均的优势是什么？

## Speaker Verification

In [ ]:
# 示例：计算 cosine score（打分），以一个 positive trial 和一个 negative trial 为例
# same-speaker trial 和 different-speaker trial 的打分方法是一样的，只是在保存文件时要记录ground truth。
def cosine_score(emb1, emb2):
    emb1 = F.normalize(emb1, dim=0)
    emb2 = F.normalize(emb2, dim=0)
    return torch.dot(emb1, emb2).item()

lookup = utterances_df.set_index("utt_id")

pos_trial = dev_trials[dev_trials["label"] == 1].iloc[0]
neg_trial = dev_trials[dev_trials["label"] == 0].iloc[0]

for name, trial in [("positive", pos_trial), ("negative", neg_trial)]:
    enroll_spk = trial["enroll_spk_id"]
    test_utt = trial["test_utt_id"]

    enroll_emb = speaker_templates[enroll_spk]
    test_path = lookup.loc[test_utt, "path"]
    test_emb = get_embedding_by_path(test_path)

    score = cosine_score(enroll_emb, test_emb)

    print(name, trial["trial_id"], "score =", score, "label =", trial["label"])


In [ ]:
# veritication trial 批量打分
lookup = utterances_df.set_index("utt_id")

def score_trials(trials_df):
    rows = []

    for _, trial in trials_df.iterrows():
        enroll_spk = trial["enroll_spk_id"]
        test_utt = trial["test_utt_id"]

        enroll_emb = speaker_templates[enroll_spk]
        test_path = lookup.loc[test_utt, "path"]
        test_emb = get_embedding_by_path(test_path)

        score = cosine_score(enroll_emb, test_emb)

        rows.append({
            **trial.to_dict(),
            "score": score,
        })

    return pd.DataFrame(rows)

dev_scores = score_trials(dev_trials)
test_scores = score_trials(test_trials)

display(dev_scores.head())
dev_scores.to_csv(OUT_DIR / "scores" / "verification_dev_scores.csv", index=False)
test_scores.to_csv(OUT_DIR / "scores" / "verification_test_scores.csv", index=False)

## Verification任务指标 EER

在 speaker verification 中，系统会为每个 trial 输出一个连续的相似度分数：$\operatorname{score}(s,t)$

分数越高，表示测试语音 $t$ 越可能来自 enrollment speaker $s$。为了把连续分数转换成 same / different 判断，需要选择一个阈值 $\tau$：

$$
\hat{y}
=
\begin{cases}
1, & \operatorname{score}(s,t) \ge \tau \\
0, & \operatorname{score}(s,t) < \tau
\end{cases}
$$

其中，$1$ 表示 same speaker，$0$ 表示 different speaker。

当阈值 $\tau$ 改变时，系统会产生两类错误：

### FAR

FAR（False Acceptance Rate）表示 different speaker 被错误接受为 same speaker 的比例：

$$
\operatorname{FAR}(\tau)
=
\frac{
\#\{ \operatorname{score}(s,t) \ge \tau,\ y = 0 \}
}{
\#\{ y = 0 \}
}
$$

FAR 越高，说明系统越容易把“不是同一个人”的 trial 错判成 same speaker。

### FRR

FRR（False Rejection Rate）表示 same speaker 被错误拒绝为 different speaker 的比例：

$$
\operatorname{FRR}(\tau)
=
\frac{
\#\{ \operatorname{score}(s,t) < \tau,\ y = 1 \}
}{
\#\{ y = 1 \}
}
$$

FRR 越高，说明系统越容易把“同一个人”的 trial 错判成 different speaker。

### EER

EER（Equal Error Rate）是指 FAR 和 FRR 尽可能相等时的错误率：

$$
\operatorname{EER}
\approx
\frac{
\operatorname{FAR}(\tau^*) + \operatorname{FRR}(\tau^*)
}{2}
$$

其中，$\tau^*$ 是使 FAR 和 FRR 最接近的阈值：

$$
\tau^*
=
\arg\min_{\tau}
\left|
\operatorname{FAR}(\tau)
-
\operatorname{FRR}(\tau)
\right|
$$

在本实验中，我们会在 dev trials 上扫描一系列候选阈值，找到近似 EER 对应的阈值 $\tau^*$。然后使用这个阈值在 test trials 上进行正式评估。

In [ ]:
# 在 dev trials 上计算 dev EER 最低时的阈值 tau，用于 test trials 正式测试
def compute_far_frr(scores, labels, thresholds):
    rows = []
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores)

    pos = labels == 1
    neg = labels == 0

    for tau in thresholds:
        pred_same = scores >= tau

        far = ((pred_same == 1) & neg).sum() / max(neg.sum(), 1)
        frr = ((pred_same == 0) & pos).sum() / max(pos.sum(), 1)

        rows.append({
            "threshold": tau,
            "FAR": far,
            "FRR": frr,
            "abs_diff": abs(far - frr),
        })

    return pd.DataFrame(rows)

thresholds = np.linspace(dev_scores["score"].min(), dev_scores["score"].max(), 300)
dev_curve = compute_far_frr(dev_scores["score"], dev_scores["label"], thresholds)

best = dev_curve.iloc[dev_curve["abs_diff"].argmin()]
tau = best["threshold"]
eer = 0.5 * (best["FAR"] + best["FRR"])

print("Dev tau:", tau)
print("Dev FAR:", best["FAR"])
print("Dev FRR:", best["FRR"])
print("Dev EER:", eer)

In [ ]:
# 绘制 dev trials 上的直方图
same_scores = dev_scores[dev_scores["label"] == 1]["score"]
diff_scores = dev_scores[dev_scores["label"] == 0]["score"]

from matplotlib import pyplot as plt

plt.figure(figsize=(7, 4))
plt.hist(same_scores, bins=20, alpha=0.6, label="same speaker")
plt.hist(diff_scores, bins=20, alpha=0.6, label="different speakers")
plt.axvline(tau, linestyle="--", label="dev threshold")
plt.xlabel("cosine score")
plt.ylabel("count")
plt.title("Verification score distribution on dev trials")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "verification_score_hist_dev.png", dpi=150)
plt.show()

In [ ]:
# 绘制 dev trials 上的FAR/FRR曲线
plt.figure(figsize=(7, 4))
plt.plot(dev_curve["threshold"], dev_curve["FAR"], label="FAR")
plt.plot(dev_curve["threshold"], dev_curve["FRR"], label="FRR")
plt.axvline(tau, linestyle="--", label="approx EER threshold")
plt.xlabel("threshold")
plt.ylabel("rate")
plt.title("FAR / FRR on dev trials")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "far_frr_curve_dev.png", dpi=150)
plt.show()

## 回答问题：
Q2. Dev EER
- 在当前数据量级下，得到这样的 EER 结果合理吗？为什么？
- 如果数据量级大一些，例如使用全量AISHELL1数据集，出现这样的 EER 结果还大吗？

In [ ]:
# 在 test trials 上应用 dev trials 上计算的阈值 tau 进行测试
def evaluate_at_threshold(score_df, tau):
    labels = score_df["label"].values.astype(int)
    scores = score_df["score"].values
    pred_same = scores >= tau

    pos = labels == 1
    neg = labels == 0

    far = ((pred_same == 1) & neg).sum() / max(neg.sum(), 1)
    frr = ((pred_same == 0) & pos).sum() / max(pos.sum(), 1)
    acc = (pred_same.astype(int) == labels).mean()

    return {
        "threshold": tau,
        "FAR": far,
        "FRR": frr,
        "accuracy": acc,
    }

test_eval = evaluate_at_threshold(test_scores, tau)
print(test_eval)

In [ ]:
# 绘制 test trials 上的直方图
same_scores = test_scores[test_scores["label"] == 1]["score"]
diff_scores = test_scores[test_scores["label"] == 0]["score"]

from matplotlib import pyplot as plt

plt.figure(figsize=(7, 4))
plt.hist(same_scores, bins=20, alpha=0.6, label="same speaker")
plt.hist(diff_scores, bins=20, alpha=0.6, label="different speakers")
plt.axvline(tau, linestyle="--", label="test threshold")
plt.xlabel("cosine score")
plt.ylabel("count")
plt.title("Verification score distribution on test trials")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "verification_score_hist_test.png", dpi=150)
plt.show()

## Speaker Identification

In [ ]:
# 闭集 speaker identification
spk_ids = sorted(speaker_templates.keys())
id_rows = []

for _, row in id_test_df[id_test_df["is_known"] == 1].iterrows():
    test_emb = get_embedding_by_path(row["path"])

    scores = []
    for spk_id in spk_ids:
        score = cosine_score(speaker_templates[spk_id], test_emb)
        scores.append(score)

    best_idx = int(np.argmax(scores))
    pred_spk = spk_ids[best_idx]

    id_rows.append({
        "utt_id": row["utt_id"],
        "true_spk": row["spk_id"],
        "pred_spk": pred_spk,
        "best_score": scores[best_idx],
        "correct": pred_spk == row["spk_id"],
    })

id_result = pd.DataFrame(id_rows)
display(id_result)

id_acc = id_result["correct"].mean()
print("Closed-set identification accuracy:", id_acc)

id_result.to_csv(OUT_DIR / "scores" / "identification_closed_set.csv", index=False)

In [ ]:
# score 矩阵热力图可视化
import matplotlib.pyplot as plt

score_mat = np.zeros((len(spk_ids), len(id_result)), dtype=np.float32)
test_ids = id_result["utt_id"].tolist()

for i, spk_id in enumerate(spk_ids):
    for j, utt_id in enumerate(test_ids):
        path = id_test_df.set_index("utt_id").loc[utt_id, "path"]
        emb = get_embedding_by_path(path)
        score_mat[i, j] = cosine_score(speaker_templates[spk_id], emb)

score_df = pd.DataFrame(score_mat, index=spk_ids, columns=test_ids)
score_df.to_csv(OUT_DIR / "scores" / "identification_score_matrix.csv")

plt.figure(figsize=(10, 5))
plt.imshow(score_mat, aspect="auto")
plt.colorbar(label="cosine score")
plt.xticks(range(len(test_ids)), test_ids, rotation=90)
plt.yticks(range(len(spk_ids)), spk_ids)
plt.xlabel("test utterance")
plt.ylabel("enrollment speaker")
plt.title("Identification score matrix")
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "identification_score_matrix.png", dpi=150)
plt.show()

In [ ]:
# 开集 speaker identification
open_rows = []

for _, row in id_test_df.iterrows():
    test_emb = get_embedding_by_path(row["path"])

    scores = []
    for spk_id in spk_ids:
        scores.append(cosine_score(speaker_templates[spk_id], test_emb))

    best_idx = int(np.argmax(scores))
    best_spk = spk_ids[best_idx]
    best_score = scores[best_idx]

    open_rows.append({
        "utt_id": row["utt_id"],
        "true_spk": row["spk_id"],
        "is_known": row["is_known"],
        "best_spk": best_spk,
        "best_score": best_score,
    })

open_df = pd.DataFrame(open_rows)
display(open_df.head())

In [ ]:
# 开集 speaker identification 的阈值 tau 来自在 verification dev trials 中的扫描
tau_id = tau

open_df["pred_open"] = open_df.apply(
    lambda r: r["best_spk"] if r["best_score"] >= tau_id else "unknown",
    axis=1,
)

display(open_df)

## 思考题
Q3. Identification 和 verification 的区别

请用自己的话说明：
- Speaker identification 关注的是什么问题？
- Speaker verification 关注的是什么问题？
- 为什么二者都可以使用 speaker embedding，但评测协议不同？

Q4. 为什么说话人验证常用 cosine similarity？

说话人验证为什么常用 cosine similarity 比较两个 embeddings？请从以下角度回答：
- embedding 是固定维度向量；
- L2 normalization 后 dot product 与 cosine similarity 的关系；
- 模型训练中常用 angular / margin-based 分类目标；（见补充阅读）
- cosine score 是否等价于概率？

Q5. 阈值选择与错误代价

假设一个系统用于门禁，另一个系统用于课堂签到：
- 哪个系统更不能接受 false accept？
- 哪个系统更不能接受 false reject？
- 这两个系统是否应该使用同一个 threshold？

Q6. 为什么不能在 test trials 上调 threshold？

- 请解释 dev set 和 test set 的区别。
- 如果你先在 test trials 上扫描 threshold，再报告该 threshold 下的 test performance，这样做有什么问题？

Q7. 为什么模型能在未见过的 AISHELL speaker上做 verification/identification 任务？

CN-Celeb checkpoint 的训练目标本质上是对训练说话人做分类，`out_n_neurons=2793`。但实验中我们拿它的 embedding 去验证/辨认 AISHELL 中未见过的说话人。请解释：
- 为什么分类训练得到的中间 embedding 可以迁移到未见过的 speaker？
- classifier head 和 embedding_model 在用途上有什么区别？
- 如果只使用 classifier 的输出类别，能否识别 AISHELL speaker？

### 补充：为什么 angular / margin-based 目标和 cosine score 有关系？

许多现代 speaker embedding 模型在训练时并不只是使用普通 softmax 分类目标，而是会使用 angular / margin-based 分类目标，例如 AM-Softmax、AAM-Softmax 或 ArcFace 类目标。

这类方法通常会对 embedding 和分类器权重做 normalization，使分类 logit 更接近于向量夹角或 cosine similarity。随后，模型会在正确类别和错误类别之间加入一个 margin，迫使同一说话人的 embeddings 更紧凑，不同说话人的 embeddings 在角度空间中分得更开。

因此，训练阶段已经鼓励模型在“方向”或“角度”上区分说话人。推理阶段使用 cosine similarity 比较两个 speaker embeddings，就和这种训练目标比较一致。

不过需要注意：cosine score 只是一个相似度分数，并不天然等价于概率。它不能直接解释为“这两段语音属于同一个人的概率”。如果需要概率意义，通常还需要额外的 calibration。

### 补充阅读

1. **Margin Matters: Towards More Discriminative Deep Neural Network Embeddings for Speaker Recognition**  
   这篇文章专门讨论 speaker recognition 中 margin-based loss 的作用，说明普通 softmax 不显式鼓励类间分离和类内紧凑，而 margin-based loss 可以学习更有区分性的 speaker embeddings。  
   https://arxiv.org/abs/1906.07317

2. **A Study on Angular Based Embedding Learning for Text-independent Speaker Verification**  
   这篇文章比较了多种 angular margin embedding learning 方法在 text-independent speaker verification 中的效果，适合理解为什么 speaker verification 常从角度空间和 cosine similarity 的角度建模。  
   https://arxiv.org/abs/1908.03990

3. **Additive Margin Softmax for Face Verification**  
   虽然这篇文章来自 face verification，但 AM-Softmax 的思想后来被大量借鉴到 speaker verification 中。它强调 feature normalization、margin 和 cosine/angular decision boundary 的关系。  
   https://arxiv.org/abs/1801.05599

4. **ArcFace: Additive Angular Margin Loss for Deep Face Recognition**  
   ArcFace 是 additive angular margin 的经典工作。它不是 speaker verification 论文，但对理解 AAM-Softmax / angular margin 的几何直觉非常有帮助。  
   https://arxiv.org/abs/1801.07698

5. **SpeechBrain loss 文档：AdditiveAngularMargin / AngularMargin**  
   SpeechBrain 中实现了 Additive Angular Margin 等 loss，并引用了 speaker recognition 相关的 margin-based embedding learning 工作。适合和本实验使用的 SpeechBrain 框架对应起来看。  
   https://speechbrain-anonym.readthedocs.io/en/latest/API/speechbrain.nnet.losses.html